In [ ]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd

In [ ]:
def import_data():
    with open("phishing.data", 'r') as f:
        data = f.read().splitlines()
    data = [list(map(int, line.split(','))) for line in data]
    X = np.array([line[:-1] for line in data])
    y = np.array([line[-1] for line in data])
    return X, y

In [ ]:
def generate_all_stumps(X_test):
    X_test = np.asarray(X_test)
    stumps = []
    for feature in range(X_test.shape[1]):
        for threshold in np.unique(X_test[:, feature]):
            for polarity in [1, -1]:
                stumps.append((feature, threshold, polarity))
    return stumps

## train

In [ ]:
def Boosting(X_train,y_train, stump_set,iterations=1000):
    X_train = np.array(X_train)
    y_train = np.array(y_train)
    m=X_train.shape[0]
    weights=np.ones(m) / m

    alphas=[]
    stumps_chosen = []


    for _ in range(iterations):
        # find the best one
        best_error=float('inf')
        best_stump=None
        best_preds=None

        # search for the best stump
        for s in stump_set:
            preds= np.ones(m)
            feature, threshold, polarity = s

            if polarity == 1:
                preds[X_train[:, feature] < threshold] = -1
            else:
                preds[X_train[:, feature] >= threshold] = -1

            error = np.sum(weights * (preds != y_train))

            if error < best_error:
                best_error = error
                best_stump = s
                best_preds = preds

        # update weights
        alpha = 0.5 * np.log((1 - best_error) / (best_error + 1e-10))
        Z = 2 * np.sqrt(best_error * (1 - best_error))

        weights *= np.exp(-alpha * y_train * best_preds) / Z

        weights /= np.sum(weights)

        alphas.append(alpha)
        stumps_chosen.append(best_stump)
    return alphas, stumps_chosen,weights

## predict

In [ ]:
def predict_boosting(X_test, alphas, stumps_chosen):
    X_test = np.array(X_test)

    final_preds = np.zeros(X_test.shape[0])

    for i in range(len(stumps_chosen)):
        feature, threshold, polarity = stumps_chosen[i]
        preds = np.ones(X_test.shape[0])

        if polarity == 1:
            preds[X_test[:, feature] < threshold] = -1
        else:
            preds[X_test[:, feature] >= threshold] = -1

        final_preds += alphas[i] * preds

    return np.sign(final_preds)

## divide and search

In [ ]:
X, y = import_data()
X_train_orig = []
X_test = []
X_val_orig = []
y_val_orig = []
y_train_orig = []
y_test = []

accuracies = []
for i in range(len(X)):
    num = np.random.rand()
    if num < 0.6:
        X_train_orig.append(X[i])
        y_train_orig.append(y[i])
    elif num < 0.8:
        X_val_orig.append(X[i])
        y_val_orig.append(y[i])
    else:
        X_test.append(X[i])
        y_test.append(y[i])
best_accuracy = 0
X_test = np.array(X_test)
y_test = np.array(y_test)
iterations = [10, 50, 100, 200, 500, 1000]
best_alphas = None
best_stump_set = None
for iter in iterations:
    alphas, stump_set, final_weights = Boosting(X_train_orig, y_train_orig, generate_all_stumps(X_train_orig), iterations=iter)

    y_val = np.array(y_val_orig)
    y_pred_val = predict_boosting(X_val_orig, alphas, stump_set)

    accuracy = np.mean(y_pred_val == y_val)
    accuracies.append(accuracy)
    print(f"Iterations: {iter}, Validation Accuracy: {accuracy*100:.2f}%")
    if accuracy > best_accuracy:
        best_accuracy = accuracy
        best_iterations = iter
        best_alphas = alphas
        best_stump_set = stump_set


plt.figure(figsize=(10, 6))
plt.plot(iterations, accuracies, marker='o')
plt.xscale('log')
plt.xlabel('Liczba iteracji')
plt.ylabel('Accuracy')
plt.title('Dokładność walidacji w zależności od liczby iteracji w AdaBoost')
plt.grid()
plt.show()

def get_confusion_matrix(y_true, y_pred):
    tp = np.sum((y_true == 1) & (y_pred == 1))
    tn = np.sum((y_true == -1) & (y_pred == -1))
    fp = np.sum((y_true == -1) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == -1))
    return tp, tn, fp, fn
y_pred = predict_boosting(X_test, best_alphas, best_stump_set)
accuracy = np.mean(y_pred == y_test)
print(f"Accuracy: {accuracy*100:.2f}%")

tp, tn, fp, fn = get_confusion_matrix(y_test, y_pred)
print(f"test size: {len(y_test)}")
print("Confusion Matrix:")
print(f"TP: {tp} | FP: {fp}")
print(f"FN: {fn} | TN: {tn}")
sns.heatmap([[tp, fp], [fn, tn]], annot=True, fmt='d', cmap='Blues', xticklabels=['Predicted Positive', 'Predicted Negative'], yticklabels=['Actual Positive', 'Actual Negative'])
plt.title('Confusion Matrix')
plt.show()

sensitivity = tp / (tp + fn)
specificity = tn / (tn + fp)
print(f"Sensitivity : {sensitivity:.2f}")
print(f"Specificity: {specificity:.2f}")